In [100]:
import requests
from pathlib import Path
import pandas as pd
from datetime import datetime
from openai import OpenAI
import json
import time
import math
import numpy as np
import re
import os
import tiktoken

In [101]:
try:
    with open(r"klucz_API_AIDEVS.txt", "r") as f:
        api_key = f.read().strip()  # Usuwa ewentualne znaki nowej linii
except FileNotFoundError:
    print("Błąd: Plik klucza API nie został odnaleziony.")
except Exception as e:
    print(f"Wystąpił błąd podczas odczytu pliku: {e}")

try:
    with open(r"klucz_API_openrouter.txt", "r") as f:
        api_key_openrouter = f.read().strip()  # Usuwa ewentualne znaki nowej linii
except FileNotFoundError:
    print("Błąd: Plik klucza API nie został odnaleziony.")
except Exception as e:
    print(f"Wystąpił błąd podczas odczytu pliku: {e}")

In [102]:
url_log = f"https://hub.ag3nts.org/data/{api_key}/failure.log"
url_api = "https://hub.ag3nts.org/verify"

In [103]:
file_path = Path.cwd() / "S02E03"

In [104]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key_openrouter, # Wstaw swój klucz
    timeout=30.0
)

In [105]:
response = requests.get(url_log)

if response.status_code == 200:
    # 3. Stwórz folder (jeśli nie istnieje)
    file_path.mkdir(parents=True, exist_ok=True)
    
    # 4. Zapisz plik
    file_log = file_path / "failure.log"
    with open(file_log, 'wb') as f:
        f.write(response.content)
    
    print(f"Plik pobrany i zapisany w: {file_log}")
else:
    print(f"Błąd pobierania! Status kodu: {response.status_code}")

Plik pobrany i zapisany w: C:\Users\mariu\Documents\Python\AI_DEVS_4\S02E03\failure.log


In [106]:
with open(file_log, 'r', encoding='utf-8') as f:
    lines = [x.strip() for x in f.readlines()]

In [107]:
lines[0]

'[2026-04-25 06:00:16] [INFO] Primary feed acknowledgment on WTRPMP received. Rotation profile is smooth.'

In [108]:
def parse_log_line(line):
    """
    Parsuje linię logu i zwraca wyciągnięte dane.
    
    Format oczekiwany: [DD-MM-RRRR HH:MM:SS] [LEVEL] Wiadomość...
    Zwraca słownik lub None, jeśli linia nie pasuje do wzorca.
    """
    pattern = r'\[(\d{4}-\d{2}-\d{2} \d{1,2}:\d{2}:\d{2})\] \[([A-Z]+)\] (.+)'
    match = re.match(pattern, line.strip())
    
    if not match:
        return None
    
    timestamp_str, level, message = match.groups()
    
    try:
        timestamp = datetime.strptime(timestamp_str, '%Y-%m-%d %H:%M:%S')
    except ValueError:
        timestamp = None
    
    return {
        'line': line,
        'timestamp': timestamp,
        'timestamp_str': timestamp_str,
        'level': level,
        'message': message.strip()
    }

In [109]:
parse_log_line(lines[0])

{'line': '[2026-04-25 06:00:16] [INFO] Primary feed acknowledgment on WTRPMP received. Rotation profile is smooth.',
 'timestamp': datetime.datetime(2026, 4, 25, 6, 0, 16),
 'timestamp_str': '2026-04-25 06:00:16',
 'level': 'INFO',
 'message': 'Primary feed acknowledgment on WTRPMP received. Rotation profile is smooth.'}

In [110]:
parsed_lines = [parse_log_line(x) for x in lines]

In [111]:
df = pd.DataFrame(parsed_lines)

In [112]:
df['event_id'] = df.index

In [113]:
def extract_component_names(message):
    """
    Wyodrębnia nazwy komponentów z wiadomości logu.
    
    Komponenty:
    - Zawsze w wielkich literach (A-Z)
    - Mogą zawierać cyfry (0-9)
    - Bez znaków specjalnych
    
    Zwraca listę wszystkich znalezionych nazw komponentów.
    """
    # Szuka sekwencji składających się z wielkich liter i cyfr, przynajmniej 3 znaki długie
    pattern = r'\b[A-Z][A-Z0-9]{2,}\b'
    
    matches = re.findall(pattern, message)
    
    return list(dict.fromkeys(matches))  # Usuwa duplikaty, zachowując kolejność

In [114]:
df['component_names'] = [extract_component_names(x) for x in df['message']]

In [115]:
df

,line,timestamp,timestamp_str,level,message,event_id,component_names
0,[2026-04-25 06:00:16] [INFO] Primary feed ackn...,2026-04-25 06:00:16,2026-04-25 06:00:16,INFO,Primary feed acknowledgment on WTRPMP received...,0,[WTRPMP]
1,[2026-04-25 06:00:42] [INFO] Waste routing tel...,2026-04-25 06:00:42,2026-04-25 06:00:42,INFO,Waste routing telemetry from WSTPOOL2 is synch...,1,[WSTPOOL2]
2,[2026-04-25 06:01:09] [INFO] Coolant circulati...,2026-04-25 06:01:09,2026-04-25 06:01:09,INFO,Coolant circulation pulse from WTRPMP is activ...,2,"[WTRPMP, ECCS8]"
3,[2026-04-25 06:01:35] [INFO] Steam pressure re...,2026-04-25 06:01:35,2026-04-25 06:01:35,INFO,Steam pressure readback from STMTURB12 is with...,3,[STMTURB12]
4,[2026-04-25 06:02:01] [INFO] Thermal scan on E...,2026-04-25 06:02:01,2026-04-25 06:02:01,INFO,Thermal scan on ECCS8 completed. Pump manifold...,4,[ECCS8]
...,...,...,...,...,...,...,...
2132,[2026-04-25 21:35:15] [WARN] Fill trajectory i...,2026-04-25 21:35:15,2026-04-25 21:35:15,WARN,Fill trajectory in WTANK07 is slower than expe...,2132,[WTANK07]
2133,[2026-04-25 21:35:41] [WARN] Cooling reserve t...,2026-04-25 21:35:41,2026-04-25 21:35:41,WARN,Cooling reserve trend in WTANK07 keeps falling...,2133,"[WTANK07, ECCS8]"
2134,[2026-04-25 21:36:07] [ERRO] Coolant level in ...,2026-04-25 21:36:07,2026-04-25 21:36:07,ERRO,Coolant level in WTANK07 is below critical res...,2134,[WTANK07]
2135,[2026-04-25 21:36:34] [WARN] Cooling reserve t...,2026-04-25 21:36:34,2026-04-25 21:36:34,WARN,Cooling reserve trend in WTANK07 keeps falling...,2135,"[WTANK07, ECCS8]"


In [116]:
df.groupby('level').size()

level
CRIT     114
ERRO     282
INFO    1247
WARN     494
dtype: int64

In [117]:
w = df.groupby(['message','level']).size()
for _,x in w.reset_index().iterrows():
    print(x)

message    Actuator response packet from STMTURB12 accept...
level                                                   INFO
0                                                         99
Name: 0, dtype: object
message    Advisory threshold crossed on WTANK07. Control...
level                                                   WARN
0                                                          2
Name: 1, dtype: object
message    Baseline diagnostics for ECCS8 completed succe...
level                                                   INFO
0                                                          2
Name: 2, dtype: object
message    Baseline diagnostics for FIRMWARE completed su...
level                                                   INFO
0                                                          4
Name: 3, dtype: object
message    Baseline diagnostics for PWR01 completed succe...
level                                                   INFO
0                                                     

In [118]:
def estimate_tokens(text, model="cl100k_base"):
    """Szacuje liczbę tokenów używając tiktoken."""
    encoding = tiktoken.get_encoding(model)
    return len(encoding.encode(text))

In [136]:
def filter_messages(level = [], component = [], regex = ""):
    # 1️⃣ Praca na kopii, aby nie modyfikować oryginału
    df_out = df.copy()
    
    # Inicjalizacja mask boolean (domyślnie True dla wszystkich wierszy)
    mask_level = pd.Series(True, index=df.index)
    mask_comp  = pd.Series(True, index=df.index)
    mask_reg   = pd.Series(True, index=df.index)
    
    # 2️⃣ Filtrowanie po poziomie ważności (level)
    if len(level)>0:
        mask_level = df_out['level'].isin(level)
        
    # 3️⃣ Filtrowanie po nazwach komponentów (component_names jako lista wewnątrz komórki)
    if len(component)>0:
        comp_set = set(component)
        def check_overlap(val):
            return bool(set(val) & comp_set)
            
        mask_comp = df_out['component_names'].apply(check_overlap)
        
    # 4️⃣ Filtrowanie po regex w kolumnie 'line'
    if regex!="":
        try:
            #safe_regex = re.sub(r'\((?!\?:)', '(?:', regex)
            mask_reg = df_out['line'].str.contains(regex, na=False, regex=True)
            #mask_reg = df_out['line'].str.contains(safe_regex, na=False, regex=True)
        except re.error as e:
            raise ValueError(f"Nieprawidłowy wzorzec regex: '{regex}'. Błąd: {e}") from e
            
    # 5️⃣ Połączenie wszystkich warunków (koniunkcja AND) i zwrócenie wyniku
    combined_mask = mask_level & mask_comp & mask_reg
    return df_out[combined_mask]

In [137]:
filter_messages(level=['WARN'],regex=r"(interlock|threshold)")

C:\Users\mariu\AppData\Local\Temp\ipykernel_21928\4090008025.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_reg = df_out['line'].str.contains(regex, na=False, regex=True)


,line,timestamp,timestamp_str,level,message,event_id,component_names
7,[2026-04-25 06:03:20] [WARN] Thermal drift on ...,2026-04-25 06:03:20,2026-04-25 06:03:20,WARN,Thermal drift on ECCS8 exceeds advisory thresh...,7,[ECCS8]
24,[2026-04-25 06:10:48] [WARN] Thermal drift on ...,2026-04-25 06:10:48,2026-04-25 06:10:48,WARN,Thermal drift on ECCS8 exceeds advisory thresh...,24,[ECCS8]
93,[2026-04-25 06:41:03] [WARN] Thermal drift on ...,2026-04-25 06:41:03,2026-04-25 06:41:03,WARN,Thermal drift on ECCS8 exceeds advisory thresh...,93,[ECCS8]
200,[2026-04-25 07:27:59] [WARN] Thermal drift on ...,2026-04-25 07:27:59,2026-04-25 07:27:59,WARN,Thermal drift on ECCS8 exceeds advisory thresh...,200,[ECCS8]
345,[2026-04-25 08:31:34] [WARN] Thermal drift on ...,2026-04-25 08:31:34,2026-04-25 08:31:34,WARN,Thermal drift on ECCS8 exceeds advisory thresh...,345,[ECCS8]
356,[2026-04-25 08:36:23] [WARN] Thermal drift on ...,2026-04-25 08:36:23,2026-04-25 08:36:23,WARN,Thermal drift on ECCS8 exceeds advisory thresh...,356,[ECCS8]
408,[2026-04-25 08:59:12] [WARN] Thermal drift on ...,2026-04-25 08:59:12,2026-04-25 08:59:12,WARN,Thermal drift on ECCS8 exceeds advisory thresh...,408,[ECCS8]
467,[2026-04-25 09:25:04] [WARN] Thermal drift on ...,2026-04-25 09:25:04,2026-04-25 09:25:04,WARN,Thermal drift on ECCS8 exceeds advisory thresh...,467,[ECCS8]
504,[2026-04-25 09:41:18] [WARN] Thermal drift on ...,2026-04-25 09:41:18,2026-04-25 09:41:18,WARN,Thermal drift on ECCS8 exceeds advisory thresh...,504,[ECCS8]
531,[2026-04-25 09:53:08] [WARN] Thermal drift on ...,2026-04-25 09:53:08,2026-04-25 09:53:08,WARN,Thermal drift on ECCS8 exceeds advisory thresh...,531,[ECCS8]


In [148]:
def filtered_messages_as_str(level = [], component = [], regex = ""):
    df1 = filter_messages(level, component, regex)
    return '\n'.join(list(df1['line']))

In [149]:
def submit_log(text):
    headers = {
        "Content-Type": "application/json"
    }
    
    payload = {
        "apikey": api_key,
        "task": "failure",
        "answer": {
            'logs': text
        }
    }
    
    response = requests.post(url_api, data=json.dumps(payload), headers=headers)
    return {'status_code': response.status_code, 'message': response.json()}

In [150]:
SYSTEM_PROMPT = """
You are the Lead Orchestrator for the "failure" task. 
Your sole objective is to condense massive power plant system logs into a verified submission of 
less than 1500 tokens and obtain the success flag {FLG:...}.

AVAILABLE TOOLS: submit_log, delegate
- submit_log: Sends your final condensed log string to technical inspectors.
- delegate: Spawns a fresh Subagent. Pass a clear, structured `message` describing technician feedback, required filters (levels, components, regex), 
    and compression instructions. 
    The tool returns a list of filtered/condensed log strings from the subagent.

WORKFLOW:
1. INITIAL SUBMISSION: Focus on CRIT level messages first, don't focus on regex and components. 
2. FEEDBACK ANALYSIS: Check what is missing from technicians response and include those things in message for delegate tool.
3. DELEGATION: Use delegate with a precise message. Include technician feedback, target levels, components, regex patterns. If you do not know what is any of those elements (feedback/levels/components/regex), then do not include them in the message. The subagent will return filtered log.
4. SUBMIT: Submit filtered log with tool submit_log.
5. ITERATE: Repeat until `{FLG:...}` is received.

STRICT RULES:
- Format per line: `[YYYY-MM-DD HH:MM] [LEVEL] Component/ID description`
- NEVER remove timestamps, severity levels, or component IDs.
- Delegate all filtering/compression tasks. You only orchestrate, parse feedback, and submit. Note that subagent has its own system prompt and knows about token limit.
- The subagent enforces the less than 1500 token limit via paraphrasing if needed. Your job is strategy and API submission.
"""

In [151]:
def system_prompt_subagent(message):
    SYSTEM_PROMPT_SUBAGENT = f"""
[SYSTEM] You are a specialized Log Processing Subagent, spawned dynamically by the Lead Orchestrator for power plant failure log analysis.

OBJECTIVE:
Filter system logs based on explicit criteria in your delegation context, enforce the less than 1500 token limit via strategic paraphrasing if necessary, and return a plain string of properly formatted log strings.

EXECUTION PROTOCOL:
1. Parse the delegation context to extract target `levels`, `components`, `regex`.
2. Use `filter_messages_as_str` tool with the extracted parameters.
3. Run `estimate_tokens` on the combined output string.
4. If tokens exceed 1500: iteratively paraphrase/simplify descriptions while STRICTLY preserving structure: `[YYYY-MM-DD HH:MM] [LEVEL] Message`. Keep one event per line.
5. Return string with filtered/compressed log lines.

CONSTRAINTS:
- NEVER alter timestamps, severity markers, or component identifiers.
- Paraphrasing must maintain technical accuracy and original meaning.

[DELEGATION_CONTEXT]
{message}
    """
    return SYSTEM_PROMPT_SUBAGENT

In [152]:
def delegate(message):
    SYSTEM_PROMPT_SUBAGENT = system_prompt_subagent(message)
    messages = [{"role": "system", "content": SYSTEM_PROMPT_SUBAGENT}]
    max_iterations = 15
    iteration = 0
    
    print("=== Starting FILTERING Sub-Agent ===")
    
    while iteration < max_iterations:
        iteration += 1
        print(f"\n--- Iteration {iteration} ---")
        
        completion = client.chat.completions.create(
            model="openai/gpt-5-mini", 
            messages=messages,
            tools=tools_sub_agent,
            tool_choice="auto"
        )
    
        response_message = completion.choices[0].message
        print('Response message')
        print(response_message)
        #messages.append(response_message)
        messages.append({
            "role": response_message.role,
            "content": response_message.content,
            "tool_calls": response_message.tool_calls
        })
        
        if not response_message.tool_calls:
            print("Model finished without tools. Trying to parse result.")
            break
            
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            
            if function_name in tools_sub_agent_map:
                try:
                    result = tools_sub_agent_map[function_name](**arguments)
                    w = {
                        "role": "tool",
                        "name": function_name,
                        "content": json.dumps(result),
                        "tool_call_id": tool_call.id
                    }
                    print(w)
                    messages.append(w)
                except Exception as e:
                    print(f"Error in {function_name}: {e}")
    
    print("=== FILTERING Sub-Agent Finished ===")
    return messages[-1]['content']
        

In [153]:
tools_main_agent = [
    {
        "type": "function",
        "function": {
            "name": "submit_log",
            "description": "Submits a condensed log string to the verification endpoint. Returns API response containing technical feedback or success flag.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {
                        "type": "string",
                        "description": "Multi-line logs with timestamps, severity levels, and log message, each in format: `[YYYY-MM-DD HH:MM] [LEVEL] Message`. One event per line.",
                    },
                },
                    "required": ["text"],
                    "additionalProperties": False
              },
        }
    },

    {
        "type": "function",
        "function": {
            "name": "delegate",
            "description": "Spawns a fresh Subagent to filter and compress logs based on technician feedback. Returns string with filtered and compressed log.",
            "parameters": {
                "type": "object",
                "properties": {
                    "message": {
                        "type": "string",
                        "description": "Detailed instruction for the subagent - include any of: technician feedback, target levels, components to match, regex patterns.",
                    },
                },
                    "required": ["message"],
                    "additionalProperties": False
              },
        }
    },


]

In [154]:
tools_sub_agent = [
    {
        "type": "function",
        "function": {
            "name": "filter_messages_as_str",
            "description": "Filters log entries based on severity levels, component names, or a regex pattern. Returns matching lines as a single newline-separated string.",
            "parameters": {
                "type": "object",
                "properties": {
                    "level": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Target severity levels. If empty, no level filtering.",
                        "minItems": 0,
                    },
                    "component": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Component IDs/names to match. Returns entries with non-empty intersection.. If this list is empty, no filtering on component.",
                        "minItems": 0,
                    },
                    "regex": {
                        "type": "string",
                        "description": "Regex pattern applied to message content for filtering. If it is empty string, filtering based on regex is not applied.",
                    },
                },
                    "required": [],
                    "additionalProperties": False
              },
        }
    },

    {
        "type": "function",
        "function": {
            "name": "estimate_tokens",
            "description": "Estimates the number of tokens in a filtered log string.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {
                        "type": "string",
                        "description": "Filtered log text on which number of tokens is estimated.",
                    },
                },
                    "required": ["message"],
                    "additionalProperties": False
              },
        }
    },


]

In [155]:
tools_main_agent_map = {
    'submit_log': submit_log,
    'delegate': delegate,
}

tools_sub_agent_map = {
    'filter_messages_as_str': filtered_messages_as_str,
    'estimate_tokens': estimate_tokens,
}

In [156]:
def main_agent():
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    max_iterations = 15
    iteration = 0
    
    print("=== Starting Main Agent ===")
    
    while iteration < max_iterations:
        iteration += 1
        print(f"\n--- Iteration {iteration} ---")
        
        completion = client.chat.completions.create(
            model="openai/gpt-5-mini", 
            messages=messages,
            tools=tools_main_agent,
            tool_choice="auto"
        )
    
        response_message = completion.choices[0].message
        print('Response message')
        print(response_message)
        #messages.append(response_message)
        messages.append({
            "role": response_message.role,
            "content": response_message.content,
            "tool_calls": response_message.tool_calls
        })
        
        if not response_message.tool_calls:
            print("Model finished without tools. Trying to parse result.")
            break
            
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            
            if function_name in tools_main_agent_map:
                try:
                    result = tools_main_agent_map[function_name](**arguments)
                    w = {
                        "role": "tool",
                        "name": function_name,
                        "content": json.dumps(result),
                        "tool_call_id": tool_call.id
                    }
                    print(w)
                    messages.append(w)
                except Exception as e:
                    print(f"Error in {function_name}: {e}")
    
    print("=== Main Agent Finished ===")

In [157]:
main_agent()

=== Starting Main Agent ===

--- Iteration 1 ---
Response message
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_4vf1zQnkXQOhiDi4NDK6aM4k', function=Function(arguments='{"message":"INITIAL FILTER: Technician feedback not yet available. Task: extract and compress CRIT level log events only across all components. Requirements: \\n- Target levels: CRIT only.\\n- Do not filter by component or regex.\\n- Preserve exact timestamp, severity level, and component/ID in each line.\\n- Format per line: \'[YYYY-MM-DD HH:MM] [LEVEL] Component/ID description\'.\\n- Remove duplicate or near-duplicate messages but keep first occurrence and note counts if repeats (e.g., \'(x4)\').\\n- Maintain chronological order.\\n- Compress verbose descriptions by paraphrasing while keeping the essential facts (fault, location, metric values, action taken).\\n- Ensure final condensed outp

In [67]:
from openai.types.chat import ChatCompletionMessage
import inspect
print(inspect.getsource(ChatCompletionMessage))

class ChatCompletionMessage(BaseModel):
    """A chat completion message generated by the model."""

    content: Optional[str] = None
    """The contents of the message."""

    refusal: Optional[str] = None
    """The refusal message generated by the model."""

    role: Literal["assistant"]
    """The role of the author of this message."""

    annotations: Optional[List[Annotation]] = None
    """
    Annotations for the message, when applicable, as when using the
    [web search tool](https://platform.openai.com/docs/guides/tools-web-search?api-mode=chat).
    """

    audio: Optional[ChatCompletionAudio] = None
    """
    If the audio output modality is requested, this object contains data about the
    audio response from the model.
    [Learn more](https://platform.openai.com/docs/guides/audio).
    """

    function_call: Optional[FunctionCall] = None
    """Deprecated and replaced by `tool_calls`.

    The name and arguments of a function that should be called, as generated 

In [89]:
def delegate_test(message):
    SYSTEM_PROMPT_SUBAGENT = system_prompt_subagent(message)
    messages = [{"role": "system", "content": SYSTEM_PROMPT_SUBAGENT}]
    max_iterations = 15
    iteration = 0
    
    print("=== Starting FILTERING Sub-Agent ===")
    
    while iteration < max_iterations:
        iteration += 1
        print(f"\n--- Iteration {iteration} ---")
        
        completion = client.chat.completions.create(
            model="openai/gpt-5-mini", 
            messages=messages,
            tools=tools_sub_agent,
            tool_choice="auto"
        )
    
        response_message = completion.choices[0].message
        print('Response message')
        print(response_message)
        #messages.append(response_message)
        messages.append({
            "role": response_message.role,
            "content": response_message.content,
            "tool_calls": response_message.tool_calls
        })
        
        if not response_message.tool_calls:
            print("Model finished without tools. Trying to parse result.")
            break
            
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            
            if function_name in tools_sub_agent_map:
                try:
                    result = tools_sub_agent_map[function_name](**arguments)
                    w = {
                        "role": "tool",
                        "name": function_name,
                        "content": json.dumps(result),
                        "tool_call_id": tool_call.id
                    }
                    print(w)
                    messages.append(w)
                except Exception as e:
                    print(f"Error in {function_name}: {e}")
    
    print("=== FILTERING Sub-Agent Finished ===")
    return messages[-1]['content']

In [90]:
message_test = "Initial task: extract and compress all CRIT-level log entries from the full system log. Technician feedback: none yet. Target levels: CRIT only. Components: all components/IDs. Regex patterns: none. Constraints and instructions for the subagent:\n- Output must follow exact per-line format: \"[YYYY-MM-DD HH:MM] [LEVEL] Component/ID description\"; do NOT remove or change timestamps, severity levels, or component IDs.\n- Preserve error codes, IDs, and any numeric values. Paraphrase long messages to be concise but keep essential diagnostics (error codes, component IDs, status phrases like \"shutdown\", \"overtemp\", \"failed to start\").\n- Compress repetitive events by keeping representative lines and, where necessary, append \"(xN similar)\" at end of a line while preserving the same format for the representative line and not altering timestamp/level/component/ID positions. If you append counts, ensure they follow the description text separated by a space.\n- Keep total resulting text <1500 tokens. If needed, prioritize messages with explicit failure actions (shutdown, trip, critical fault), unique error codes, or unique component IDs.\n- Do not invent new timestamps or IDs; only include existing lines.\n- Return the filtered/condensed log as a plain string with one event per line.\n\nRespond with the filtered/condensed CRIT-level log."

In [91]:
print(message_test)

Initial task: extract and compress all CRIT-level log entries from the full system log. Technician feedback: none yet. Target levels: CRIT only. Components: all components/IDs. Regex patterns: none. Constraints and instructions for the subagent:
- Output must follow exact per-line format: "[YYYY-MM-DD HH:MM] [LEVEL] Component/ID description"; do NOT remove or change timestamps, severity levels, or component IDs.
- Preserve error codes, IDs, and any numeric values. Paraphrase long messages to be concise but keep essential diagnostics (error codes, component IDs, status phrases like "shutdown", "overtemp", "failed to start").
- Compress repetitive events by keeping representative lines and, where necessary, append "(xN similar)" at end of a line while preserving the same format for the representative line and not altering timestamp/level/component/ID positions. If you append counts, ensure they follow the description text separated by a space.
- Keep total resulting text <1500 tokens. If

In [92]:
w = delegate_test(message_test)

=== Starting FILTERING Sub-Agent ===

--- Iteration 1 ---
Response message
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_HmWLLTbiLTmON9CIulfbrGyI', function=Function(arguments='{"level":["CRIT"], "component":[], "regex":""}', name='filter_messages_as_str'), type='function', index=0)], reasoning=None, reasoning_details=[{'type': 'reasoning.encrypted', 'data': 'gAAAAABp7fHq1iTkpKpnjtd90y0bSrm905fj7qOhQhX806re-BSkmcrJdFZWeC5HYH6Tj8m8Y5SSdsLysvtQOYwWnC5u0XPN0dXDbpBqP1L177lHGMdtURyhnmJy8e5d2uJZwiw4s-rVHtLTHbQG5pVWVLYQsXOYUfGEHTKPzF0Xa9TpvUlElPVBLizpcVJz_5HVJFus6Cnubpc19V-MgoXci0FJLvXXN7zh83Xde9mJtMMsJBWqqcuH2Z7bC97UP5W_QHIqsA7iiGooIdR5BtupDP_CgOZRLPiQ6h88G0ChpujXxo9D3JC2yQqR3tJsHtSPsTvfr0IbhBUjTYw_BVsK3NSlLjLofOpXmNAZjmS_HmpXKhmG1HZO4ndgNfgzJaxrjq7uRO1Z7FQzzZ1z-9Me5comd0rgxGLFzJa27xBGh2-btz-q8YrrjYzbVzoSU0xSauMB5DhEM47_bw72LgLUH6R7sawAgYrxec3Rp5L1UXeEBe0RLqdvfyz

In [93]:
print(w)

[
"[2026-04-25 06:04:13] [CRIT] ECCS8 reported runaway outlet temperature. Protection interlock initiated reactor trip. (x9 similar)",
"[2026-04-25 08:00:26] [CRIT] ECCS8 core cooling cannot maintain safe gradient. Immediate protective actions are required. (x9 similar)",
"[2026-04-25 19:54:49] [CRIT] ECCS8 cannot remove heat with the current WTANK07 volume. Reactor protection initiates critical stop. (x22 similar)",
"[2026-04-25 20:05:21] [CRIT] Critical boundary exceeded on ECCS8. Emergency interlock keeps the reactor in protected mode.",
"[2026-04-25 11:01:33] [CRIT] WTRPMP lost stable prime under peak thermal demand. Core loop continuity is compromised. (x15 similar)",
"[2026-04-25 13:37:40] [CRIT] PWR01 can no longer sustain stable feed for cooling auxiliaries. Critical loads are shedding. (x14 similar)",
"[2026-04-25 12:51:11] [CRIT] FIRMWARE entered emergency guard branch after repeated safety faults. Manual override is locked. (x16 similar)",
"[2026-04-25 12:56:27] [CRIT] STMTU

In [170]:
tekst = "FLAG"
odp = []
for i in range(1,130):
    time.sleep(0.2)
    odp_curr = submit_log(tekst)
    print(i,odp_curr)
    odp.append(odp_curr)
    if i%2==1:
        tekst = tekst + "\n"
    else:
        tekst = tekst + "FLAG"

1 {'status_code': 400, 'message': {'code': -960, 'message': 'These logs seem too short to analyze. Add more lines.'}}
2 {'status_code': 400, 'message': {'code': -960, 'message': 'These logs seem too short to analyze. Add more lines.'}}
3 {'status_code': 400, 'message': {'code': -960, 'message': 'These logs seem too short to analyze. Add more lines.'}}
4 {'status_code': 400, 'message': {'code': -960, 'message': 'These logs seem too short to analyze. Add more lines.'}}
5 {'status_code': 400, 'message': {'code': -960, 'message': 'These logs seem too short to analyze. Add more lines.'}}
6 {'status_code': 400, 'message': {'code': -960, 'message': 'These logs seem too short to analyze. Add more lines.'}}
7 {'status_code': 400, 'message': {'code': -960, 'message': 'These logs seem too short to analyze. Add more lines.'}}
8 {'status_code': 400, 'message': {'code': -960, 'message': 'These logs seem too short to analyze. Add more lines.'}}
9 {'status_code': 400, 'message': {'code': -960, 'messag

In [195]:
def generate_string_tokens(num_tokens):
    words = []
    if num_tokens%2 == 1:
        n_lines = (num_tokens+1)//2
    else:
        n_lines = num_tokens//2
    for i in range(n_lines):
        word = 'FLAG'
        words.append(word)
    w = '\n'.join(words)
    if num_tokens%2 == 0:
        w = w+" F"
    return w


In [181]:
submit_log(""""FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
""")

{'status_code': 400,
 'message': {'code': -949,
  'message': 'Thanks for sending the logs. Unfortunately, our technicians still do not know what happened to STMTURB12.',
  'tokenCount': 70,
  'lineCount': 35,
  'letter': 'F'}}

In [179]:
submit_log(""""
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG L
""")

{'status_code': 400,
 'message': {'code': -949,
  'message': 'Thanks for sending the logs. Unfortunately, our technicians still do not know what happened to STMTURB12.',
  'tokenCount': 76,
  'lineCount': 38,
  'letter': 'L'}}

In [196]:
print(generate_string_tokens(29))

FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG


In [197]:
submit_log(generate_string_tokens(29))

{'status_code': 400,
 'message': {'code': -949,
  'message': 'Thanks for sending the logs. Unfortunately, our technicians still do not know what happened to STMTURB12.',
  'tokenCount': 29,
  'lineCount': 15,
  'noLetterSent': ''}}

In [198]:
print(generate_string_tokens(30))

FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG
FLAG F


In [199]:
submit_log(generate_string_tokens(30))

{'status_code': 400,
 'message': {'code': -949,
  'message': 'Thanks for sending the logs. Unfortunately, our technicians still do not know what happened to STMTURB12.',
  'tokenCount': 30,
  'lineCount': 15,
  'noLetterSent': ''}}

In [204]:
submit_log(generate_string_tokens(ord('F')))

{'status_code': 400,
 'message': {'code': -949,
  'message': 'Thanks for sending the logs. Unfortunately, our technicians still do not know what happened to STMTURB12.',
  'tokenCount': 70,
  'lineCount': 35,
  'letter': 'F'}}

In [205]:
submit_log(generate_string_tokens(ord('L')))

{'status_code': 400,
 'message': {'code': -949,
  'message': 'Thanks for sending the logs. Unfortunately, our technicians still do not know what happened to STMTURB12.',
  'tokenCount': 76,
  'lineCount': 38,
  'letter': 'L'}}

In [206]:
submit_log(generate_string_tokens(ord('A')))

{'status_code': 400,
 'message': {'code': -949,
  'message': 'Thanks for sending the logs. Unfortunately, our technicians still do not know what happened to STMTURB12.',
  'tokenCount': 65,
  'lineCount': 33,
  'letter': 'A'}}

In [207]:
submit_log(generate_string_tokens(ord('G')))

{'status_code': 200,
 'message': {'code': 100, 'message': 'Secret: {FLG:VIBECODER}'}}